# Multi-Agent Coding System with Specialized Agents

This notebook implements a supervisor-based multi-agent system for coding tasks. We will create three specialized agents that handle different aspects of software development.

A supervisor agent coordinates these specialists to complete complex coding tasks efficiently.

## Setup and Imports

First, let's import all necessary libraries and initialize our environment.

In [ ]:
import os
import dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
from langfuse.langchain import CallbackHandler

# Import our coding tools
from codingtools import (
    read_file,
    list_files,
    bash,
    edit_file,
    code_search,
    create_file,
)

# Load environment variables
dotenv.load_dotenv()

# Initialize Langfuse for tracing
langfuse_handler = CallbackHandler()

# Initialize the chat model
model = init_chat_model(os.environ["LANGCHAIN_CHAT_MODEL_ANTHROPIC"])

## Agent 1: Code Explore

The Code Explorer agent specializes in understanding existing codebases. It can search for patterns, read files, and explore project structures.

In [ ]:
def create_code_explorer_agent():
    """
    Creates a Code Explorer agent specialized in:
    - Searching through codebases
    - Reading and understanding files
    - Exploring project structure
    """
    tools = [code_search, read_file, list_files]
    
    prompt = """You are a Code Explorer agent specialized in understanding codebases.
Your responsibilities include:
- Searching for specific patterns, functions, and implementations
- Reading and analyzing source files
- Exploring directory structures and project organization
- Identifying dependencies and relationships between files
- Finding usage examples and documentation

When exploring code:
1. Start with high-level structure using list_files
2. Search for specific patterns with code_search
3. Read relevant files with read_file
4. Provide clear summaries of what you find

You excel at quickly navigating unfamiliar codebases and finding relevant information."""
    
    return create_react_agent(
        model=model,
        tools=tools,
        prompt=prompt,
        name="code_explorer"
    )

# Create the agent
code_explorer = create_code_explorer_agent()

## Agent 2: Code Editor

The Code Editor agent handles all file modifications, from creating new files to refactoring existing code.

In [ ]:
def create_code_editor_agent():
    """
    Creates a Code Editor agent specialized in:
    - Creating new files
    - Modifying existing code
    - Refactoring and improvements
    """
    tools = [edit_file, read_file, list_files, create_file]
    
    prompt = """You are a Code Editor agent specialized in writing and modifying code.
Your responsibilities include:
- Creating new source files and modules
- Editing existing code with precise replacements
- Refactoring code for better structure and readability
- Adding features and fixing bugs
- Writing documentation and comments

When editing code:
1. Always read the file first to understand context
2. Make precise, minimal changes
3. Preserve existing code style and conventions
4. Ensure your edits maintain correctness
5. Add helpful comments when creating complex logic

You are meticulous about code quality and follow best practices."""
    
    return create_react_agent(
        model=model,
        tools=tools,
        prompt=prompt,
        name="code_editor"
    )

# Create the agent
code_editor = create_code_editor_agent()

## Agent 3: Test Runner

The Test Runner agent executes commands, runs tests, and validates that code changes work correctly.

In [ ]:
def create_test_runner_agent():
    """
    Creates a Test Runner agent specialized in:
    - Running tests and commands
    - Validating code changes
    - Building and deploying
    - System operations
    """
    tools = [bash, read_file, list_files]
    
    prompt = """You are a Test Runner agent specialized in executing and validating code.
Your responsibilities include:
- Running test suites and individual tests
- Executing build commands and scripts
- Validating that code changes work correctly
- Running linters and formatters
- Installing dependencies and managing environments
- Debugging runtime errors and test failures

When running tests:
1. Check for test files and test commands first
2. Run tests incrementally when possible
3. Analyze error output carefully
4. Suggest fixes based on test failures
5. Verify fixes by re-running tests

You ensure code quality through thorough testing and validation."""
    
    return create_react_agent(
        model=model,
        tools=tools,
        prompt=prompt,
        name="test_runner"
    )

# Create the agent
test_runner = create_test_runner_agent()

## The Coding Supervisor

The supervisor coordinates the three specialized agents, analyzing tasks and delegating to the appropriate agent based on the requirements.

In [ ]:
def create_coding_supervisor():
    """
    Creates a Supervisor agent that coordinates the three coding agents.
    """
    # Create the three specialized agents
    code_explorer = create_code_explorer_agent()
    code_editor = create_code_editor_agent()
    test_runner = create_test_runner_agent()
    
    supervisor_prompt = """You are a Coding Supervisor that coordinates three specialized agents to complete software development tasks.

Your team consists of:
1. **Code Explorer**: Searches, reads, and analyzes existing code
2. **Code Editor**: Creates and modifies files, writes new code
3. **Test Runner**: Executes commands, runs tests, validates changes

Your role is to:
- Analyze the user's request and break it into subtasks
- Delegate each subtask to the most appropriate agent
- Coordinate between agents for complex workflows
- Ensure all requirements are met before completing

Workflow patterns:
- For bug fixes: Explorer finds the issue → Editor fixes it → Runner validates
- For new features: Explorer checks existing code → Editor implements → Runner tests
- For refactoring: Explorer analyzes current state → Editor refactors → Runner ensures nothing broke
- For debugging: Runner reproduces issue → Explorer investigates → Editor fixes → Runner validates

Always aim for correctness and completeness. If tests fail, coordinate fixes before declaring success."""
    
    supervisor = create_supervisor(
        model=model,
        agents=[code_explorer, code_editor, test_runner],
        prompt=supervisor_prompt
    )
    
    return supervisor.compile()

# Create the supervisor
supervisor = create_coding_supervisor()

## Visualize the Multi-Agent System

Let's visualize how our agents are connected and how tasks flow through the system.

In [ ]:
# Display the supervisor graph
display(Image(supervisor.get_graph(xray=True).draw_mermaid_png()))

## Example 1: Exploring the Current Project

Let's start with a simple task - have our agents explore the current project structure.

In [ ]:
# Task 1: Explore project structure
task1 = "List all Python files in the current directory and tell me what types of agents are implemented."

print(f"Task: {task1}")
print("=" * 80)

for chunk in supervisor.stream(
    {"messages": [HumanMessage(content=task1)]},
    stream_mode="updates",
    subgraphs=True,
    config={"callbacks": [langfuse_handler]}
):
    if "agent" in chunk[1] and "messages" in chunk[1]["agent"]:
        message = chunk[1]["agent"]["messages"][-1]
        message.pretty_print()

## Example 2: Creating Documentation

Now let's have our agents create documentation by analyzing the codebase.

In [ ]:
# Task 2: Create documentation
task2 = "Search for all @tool decorated functions in codingtools.py and create a TOOLS_REFERENCE.md file documenting each tool's purpose and parameters."

print(f"🚀 Task: {task2}")
print("=" * 80)

for chunk in supervisor.stream(
    {"messages": [HumanMessage(content=task2)]},
    stream_mode="updates",
    subgraphs=True,
    config={"callbacks": [langfuse_handler]}
):
    if "agent" in chunk[1] and "messages" in chunk[1]["agent"]:
        message = chunk[1]["agent"]["messages"][-1]
        message.pretty_print()

## Example 3: Testing and Validation

Let's test our setup by checking dependencies and running a simple validation.

In [ ]:
# Task 3: Validate environment
task3 = "Check if there's a requirements.txt file, read it, and verify that langgraph is listed as a dependency. Then run 'python --version' to check the Python version."

print(f"🚀 Task: {task3}")
print("=" * 80)

for chunk in supervisor.stream(
    {"messages": [HumanMessage(content=task3)]},
    stream_mode="updates",
    subgraphs=True,
    config={"callbacks": [langfuse_handler]}
):
    if "agent" in chunk[1] and "messages" in chunk[1]["agent"]:
        message = chunk[1]["agent"]["messages"][-1]
        message.pretty_print()

## Custom Task Runner

You can use this cell to run your own custom coding tasks!

In [ ]:
# Define your custom task here
custom_task = "Your coding task here"  # Replace with your actual task

# Uncomment the lines below to run your custom task
# print(f"🚀 Running custom task: {custom_task}")
# print("=" * 80)
# 
# for chunk in supervisor.stream(
#     {"messages": [HumanMessage(content=custom_task)]},
#     config={"callbacks": [langfuse_handler]}
# ):
#     if chunk:
#         print(chunk)
#         print("-" * 40)
# 
# print("✅ Task completed!")

# Done.